In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load the dataset we just saved
df = pd.read_csv("/Users/abhishek/Desktop/world-bank-indicators-project/final_plfs_with_exposure.csv")

In [2]:
# Keep only working-age (15-59) employed people who have a positive salaried wage
df_wage = df[(df['Age'] >= 15) & (df['Age'] <= 59) & (df['CWS_Earnings_Salaried'] > 0)].copy()
print(f"Rows kept for wage analysis: {len(df_wage):,}")

Rows kept for wage analysis: 132,457


In [3]:
# 1. Create a Female dummy (1 if female, 0 if male). In PLFS, Sex=2 is Female.
df_wage['Female'] = (df_wage['Sex'] == 2).astype(int)

# 2. Create a Log Wage variable (we take log because wages are highly skewed)
df_wage['Log_Wage'] = np.log(df_wage['CWS_Earnings_Salaried'])

# 3. Create a Post dummy (2024 = 1, 2022-2023 = 0) to fulfill the assignment's Post*treatment requirement
df_wage['Post'] = (df_wage['Year'] == 2024).astype(int)

print(df_wage[['Female', 'Log_Wage', 'Post']].describe())

              Female       Log_Wage           Post
count  132457.000000  132457.000000  132457.000000
mean        0.255570       9.660281       0.342919
std         0.436183       0.790115       0.474686
min         0.000000       5.703782       0.000000
25%         0.000000       9.159047       0.000000
50%         0.000000       9.615805       0.000000
75%         1.000000      10.239960       1.000000
max         1.000000      13.161584       1.000000


# Does higher routine exposure affect women's wages differently than men's?

In [9]:
# 1. Find states with at least 100 female salaried workers
# This prevents the "rank" error by dropping tiny states with 0 or 1 women
state_counts = df_wage[df_wage['Female'] == 1].groupby('State_UT_Code').size()
big_states = state_counts[state_counts >= 100].index

# 2. Filter the data
df_wage_clean = df_wage[df_wage['State_UT_Code'].isin(big_states)].copy()
print(f"Filtered to {len(df_wage_clean):,} rows across {len(big_states)} major states.")

# 3. Run the model again on the clean data
model1_clean = smf.ols(
    formula="Log_Wage ~ Routine_Intensity * Female + C(State_UT_Code) + C(Year)",
    data=df_wage_clean
).fit(cov_type='cluster', cov_kwds={'groups': df_wage_clean['State_UT_Code']})

# 4. Print just the variables we care about using standard pandas dataframe
results = pd.DataFrame({
    'Coefficient': model1_clean.params,
    'Std Error': model1_clean.bse,
    'P-value': model1_clean.pvalues
})
print(results.loc[['Intercept', 'Routine_Intensity', 'Female', 'Routine_Intensity:Female']])

Filtered to 131,834 rows across 34 major states.
                          Coefficient  Std Error       P-value
Intercept                   10.019157   0.015349  0.000000e+00
Routine_Intensity           -0.587453   0.057397  1.384344e-24
Female                      -0.381101   0.025895  5.000023e-49
Routine_Intensity:Female    -0.346860   0.067781  3.097874e-07


In [10]:
# --- Model 2: Urban vs Rural Split ---
# Sector 1 = Rural, Sector 2 = Urban
urban = df_wage_clean[df_wage_clean['Sector'] == 2]
rural = df_wage_clean[df_wage_clean['Sector'] == 1]

model_urban = smf.ols("Log_Wage ~ Routine_Intensity * Female + C(State_UT_Code) + C(Year)", data=urban).fit(cov_type='cluster', cov_kwds={'groups': urban['State_UT_Code']})
model_rural = smf.ols("Log_Wage ~ Routine_Intensity * Female + C(State_UT_Code) + C(Year)", data=rural).fit(cov_type='cluster', cov_kwds={'groups': rural['State_UT_Code']})


In [15]:
# Load ONLY the education column from the raw files
edu22 = pd.read_csv("/Users/abhishek/Desktop/world-bank-indicators-project/PLFS_2022/cperv1.csv", usecols=['b4q8_cperv1'])['b4q8_cperv1']
edu23 = pd.read_csv("/Users/abhishek/Desktop/world-bank-indicators-project/PLFS_2023/cperv1.csv", usecols=['b4q8_cperv1'])['b4q8_cperv1']
edu24 = pd.read_csv("/Users/abhishek/Desktop/world-bank-indicators-project/PLFS_2024/cperv1.csv", usecols=['General_Education_Level'])['General_Education_Level']

# Combine them
edu_all_raw = pd.concat([edu22, edu23, edu24], ignore_index=True)

# Load the FINAL file (which has Routine_Intensity)
plfs_final = pd.read_csv("/Users/abhishek/Desktop/world-bank-indicators-project/final_plfs_with_exposure.csv")

# Add education directly by row number (since we never sorted the data)
plfs_final['Education'] = edu_all_raw

# Filter to our wage sample
df_wage_clean = plfs_final[(plfs_final['Age'] >= 15) & (plfs_final['Age'] <= 59) & (plfs_final['CWS_Earnings_Salaried'] > 0)].copy()
state_counts = df_wage_clean[df_wage_clean['Sex'] == 2].groupby('State_UT_Code').size()
big_states = state_counts[state_counts >= 100].index
df_wage_clean = df_wage_clean[df_wage_clean['State_UT_Code'].isin(big_states)].copy()

# Re-create the model variables
df_wage_clean['Female'] = (df_wage_clean['Sex'] == 2).astype(int)
df_wage_clean['Log_Wage'] = np.log(df_wage_clean['CWS_Earnings_Salaried'])
df_wage_clean['Low_Skill'] = (df_wage_clean['Education'] <= 7).astype(int)

print("Data prepared! Routine_Intensity is in the data.")

Data prepared! Routine_Intensity is in the data.


In [16]:
# Re-create the model variables
df_wage_clean['Female'] = (df_wage_clean['Sex'] == 2).astype(int)
df_wage_clean['Log_Wage'] = np.log(df_wage_clean['CWS_Earnings_Salaried'])

# PLFS Education codes: 01 to 07 = Below Secondary, 08 to 13 = Secondary and above
# We will treat 1-7 as Low Skill (1), and 8+ as High Skill (0)
df_wage_clean['Low_Skill'] = (df_wage_clean['Education'] <= 7).astype(int)

print(df_wage_clean['Low_Skill'].value_counts())

Low_Skill
1    87699
0    44135
Name: count, dtype: int64


In [17]:
# Split into Low Skill and High Skill
low_skill = df_wage_clean[df_wage_clean['Low_Skill'] == 1]
high_skill = df_wage_clean[df_wage_clean['Low_Skill'] == 0]

# Run the models
model_low = smf.ols("Log_Wage ~ Routine_Intensity * Female + C(State_UT_Code) + C(Year)", data=low_skill).fit(cov_type='cluster', cov_kwds={'groups': low_skill['State_UT_Code']})
model_high = smf.ols("Log_Wage ~ Routine_Intensity * Female + C(State_UT_Code) + C(Year)", data=high_skill).fit(cov_type='cluster', cov_kwds={'groups': high_skill['State_UT_Code']})

# Print the interaction results
print("--- LOW SKILL (Below Secondary) ---")
print(f"Coef: {model_low.params['Routine_Intensity:Female']:.4f} | P-value: {model_low.pvalues['Routine_Intensity:Female']:.4f}")

print("\n--- HIGH SKILL (Secondary+) ---")
print(f"Coef: {model_high.params['Routine_Intensity:Female']:.4f} | P-value: {model_high.pvalues['Routine_Intensity:Female']:.4f}")

--- LOW SKILL (Below Secondary) ---
Coef: -0.3192 | P-value: 0.0000

--- HIGH SKILL (Secondary+) ---
Coef: -0.4024 | P-value: 0.0000


In [18]:


# Save all the key results into a clean table for your memo
results_table = pd.DataFrame({
    'Model': ['Baseline', 'Urban', 'Rural', 'Low Skill', 'High Skill'],
    'Interaction_Coef': [-0.3468, model_urban.params['Routine_Intensity:Female'], 
                         model_rural.params['Routine_Intensity:Female'], 
                         model_low.params['Routine_Intensity:Female'], 
                         model_high.params['Routine_Intensity:Female']],
    'P_Value': [0.0000, model_urban.pvalues['Routine_Intensity:Female'], 
                model_rural.pvalues['Routine_Intensity:Female'], 
                model_low.pvalues['Routine_Intensity:Female'], 
                model_high.pvalues['Routine_Intensity:Female']]
})

import os
os.makedirs("/Users/abhishek/Desktop/world-bank-indicators-project/outputs/tables", exist_ok=True)
results_table.to_csv("/Users/abhishek/Desktop/world-bank-indicators-project/outputs/tables/task7_results.csv", index=False)
print(results_table)

        Model  Interaction_Coef       P_Value
0    Baseline         -0.346800  0.000000e+00
1       Urban         -0.450644  8.168318e-14
2       Rural         -0.194027  5.250397e-02
3   Low Skill         -0.319177  4.292753e-06
4  High Skill         -0.402436  2.859285e-08
